## Intersect differentially targeted (DT) small RNA peaks with differentially expressed (DE) genes
The data for the differentially expressed genes are published in this paper: <br>
Wolfe et al. 2023. doi: https://doi.org/10.1111/mec.17070

### Load required libraries


In [ ]:
setwd("/home/mier0006/Documents/phd_dact/smRNA_dact/")

In [3]:
library("tidyverse")
library("ggplot2")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Print session info

In [4]:
sessionInfo()

R version 4.5.0 (2025-04-11)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 24.04.2 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.12.0 
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.12.0  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Stockholm
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.4 forcats_1.0.0   stringr_1.5.1   dplyr_1.1.4    
 [5] purrr_1.0.4     readr_2.1.5     tidyr_1.3.1     tibble_3.2.1   
 [9] ggplot2_3.5.2   tidyverse_2.0.0

loaded via a nam

Load DT small RNA targeted genes
- extract peaks with a FDR corrected p-value <= 0.05
- keep only the columns: <br>
    peakID, geneID, region, logFC
- rename logFC to logFC_dt

In [48]:
dt <- read.table("04_DT-peaks_classification/genes-DTpeaks_tm_20to24nt.txt", header=T, sep="\t") %>% 
  filter((FDR<=0.05)) %>%
  select(peakID, geneID, region, logFC) %>%
  rename(logFC_dt = "logFC")
head(dt)

,peakID,geneID,region,logFC_dt
,<chr>,<chr>,<chr>,<dbl>
1,scaffold1_42451085:17577101-17577400,Dinc060395,intron,2.222204
2,scaffold1_42451085:30407201-30407300,Dinc060660,exon,-4.650598
3,scaffold1_42451085:32606201-32606300,Dinc00371,intron,-3.541465
4,scaffold1_42451085:622301-622900,Dinc00020,exon,1.442233
5,scaffold1_42451085:622301-622900,Dinc00020,1000bp_downstream,1.442233
6,scaffold1_42451085:622301-622900,Dinc060024,exon,1.442233


Load DE gene data frame

- make a new column with a modified `genes` column to match the format of the `geneID` column in the DT small RNA data frame
- remove the `genes` column
- remove duplicate rows
- rename `logFC` to `logFC_DE`

In [49]:
de <- read.table("06_RNAseq_overlapp/DE-genes_geneID-FC_TMW.tab", header=T, sep="\t") %>% 
  mutate(geneID = gsub("-.*", "", genes)) %>% 
  select(!genes)  %>% 
  distinct() %>% 
  rename(logFC_de = "logFC")
head(de)


,logFC_de,geneID
,<dbl>,<chr>
1,-0.8132844,Dinc061141
2,-0.5540832,Dinc080279
3,-1.5900623,Dinc082500
4,-1.1877020,Dinc082807
5,1.4561736,Dinc086527
6,-1.8659226,Dinc086672


Load data containing information about genomic interactions
- keep only the columns: <br>
peakID, effMAJ, effTRA
- keep only peaks in the DT small RNA data frame

In [50]:
gi <- read.table("03_genomic-intercations/genes-DTpeaks_4x-vs-2x_20to24nt.txt", header=T, sep="\t") %>%
  select(peakID, effMAJ, effTRA) %>%
  filter(peakID %in% dt$peakID)
head(gi)

,peakID,effMAJ,effTRA
,<chr>,<chr>,<chr>
1,scaffold1_42451085:17577101-17577400,NA,domI_down
2,scaffold1_42451085:30407201-30407300,transgressive_up,NA
3,scaffold1_42451085:622301-622900,NA,domI_down
4,scaffold1_42451085:622301-622900,NA,domI_down
5,scaffold1_42451085:622301-622900,NA,domI_down
6,scaffold1_42451085:7240201-7240300,domF_down,domI_up


Combine all data frames

In [44]:
df  <- merge(dt, de) %>% 
  merge(., gi, all=T) %>%
  filter(!(is.na(logFC_de))) %>%
  rename(target_region = "region") %>%
  distinct()
df

peakID,geneID,target_region,logFC_dt,logFC_de,effMAJ,effTRA
<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>
scaffold105_8032468:1393201-1393700,Dinc091776,exon,-3.528605,-1.4540006,NA,domI_up
scaffold105_8032468:1393801-1394600,Dinc091776,exon,-4.353099,-1.4540006,NA,domI_up
scaffold105_8032468:1394701-1395000,Dinc091776,exon,-5.757077,-1.4540006,NA,domI_up
scaffold106_7996922:6416201-6416400,Dinc083288,intron,-3.676176,-1.0006479,domI_down,NA
scaffold109_7658945:2124501-2125100,Dinc077269,intron,-1.022087,-0.5097641,domF_down,NA
scaffold176_5804863:3977801-3977900,Dinc106626,intron,-3.325999,0.7186814,domI_down,NA
scaffold226_4712730:4313701-4313900,Dinc108497,intron,-7.406952,-0.6311271,transgressive_up,NA
scaffold233_4568996:2215401-2215500,Dinc097501,exon,4.396770,-0.8048408,NA,transgressive_up
scaffold306_3274172:520301-520400,Dinc123280,intron,-4.756203,0.8795493,transgressive_up,NA


Count number of DT peaks in each intersected DE gene
- extract  the columns: <br>
    peakID, geneID
- group by `geneID`
- count the number of peaks

In [ ]:
df %>%
select(geneID, peakID) %>% 
group_by(geneID) %>% 
summarise(n = n())

geneID,n
<chr>,<int>
Dinc069138,1
Dinc072575,1
Dinc077269,1
Dinc083288,1
Dinc091776,3
Dinc097501,1
Dinc102099,1
Dinc106626,1
Dinc108497,1
